# Data Cleaning & Preprocessing Pipeline

Notebook untuk membersihkan dan memproses data sanadset.csv

## Import Library

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast
import re
import os
from collections import Counter

print("✅ Libraries imported successfully")

## Load Data

In [ ]:
# Load dataset
df = pd.read_csv("../data/raw/sanadset.csv", encoding='utf-8', header=None)
df.columns = ['Hadith', 'Book', 'Num_hadith', 'Matn', 'Sanad', 'Sanad_Length']

# Pastikan kolom Sanad ada
assert 'Sanad' in df.columns, "❌ Kolom 'Sanad' tidak ditemukan!"

print(f"✅ Data loaded: {len(df)} baris")
print(f"   Kolom: {df.columns.tolist()}")
print(f"   Tipe data: {df.dtypes.to_dict()}")

print(f"\n📊 Sample data:")
print(df.head(3))

## Filter Data

In [ ]:
# Hapus baris tanpa sanad
print(f"📊 Before filtering: {len(df)} rows")

df = df[df['Sanad'] != 'No SANAD'].copy()
print(f"📊 After removing 'No SANAD': {len(df)} rows")

# Filter out rows where Sanad is empty or invalid
df = df[df['Sanad'].notna()].copy()
print(f"📊 After removing null Sanad: {len(df)} rows")

print(f"\n✅ Data filtered successfully")

## Convert Sanad to List

In [ ]:
def safe_literal_eval(val):
    """Safely convert string representation of list to actual list"""
    try:
        return ast.literal_eval(val)
    except:
        return val

# Convert Sanad string to list
print("🔄 Converting Sanad from string to list...")
if isinstance(df['Sanad'].iloc[0], str):
    df['Sanad'] = df['Sanad'].apply(safe_literal_eval)
    print("✅ Conversion completed")

print(f"📊 Sample Sanad after conversion:")
print(f"   Type: {type(df['Sanad'].iloc[0])}")
print(f"   Length: {len(df['Sanad'].iloc[0])}")
print(f"   Content: {df['Sanad'].iloc[0]}")

## Remove Harakat (Arabic Diacritics)

In [ ]:
def remove_harakat(text):
    """Remove Arabic diacritics (harakat)"""
    pattern = r'[\u064B-\u065F\u0670\u06D6-\u06ED]'
    
    if isinstance(text, list):
        return [re.sub(pattern, '', t) for t in text]
    elif isinstance(text, str):
        return re.sub(pattern, '', text)
    return text

print("🧹 Removing harakat from Sanad...")
df['Sanad_no_harakat'] = df['Sanad'].apply(remove_harakat)
print("✅ Harakat removal completed")

print(f"📊 Sample comparison:")
print(f"   Original: {df['Sanad'].iloc[0]}")
print(f"   No harakat: {df['Sanad_no_harakat'].iloc[0]}")

## Filter Specific Words

In [ ]:
# Define kinship words to filter out
kinship_words = [
    'ابيه', 'أبيه', 'اباه', 'أباه', 'ابو', 'أبو',
    'اخيه', 'أخيه', 'امه', 'أمه', 'عمه', 'عمه'
]

print(f"🚫 Filtering out kinship words: {kinship_words}")
print(f"📊 Before filtering: {len(df)} rows")

# Remove rows containing kinship words
df = df[
    ~df['Sanad_no_harakat'].apply(
        lambda lst: any(word in lst for word in kinship_words) if isinstance(lst, list) else False
    )
].copy()

print(f"📊 After filtering kinship words: {len(df)} rows")
print("✅ Kinship word filtering completed")

## Clean and Normalize Text

In [ ]:
def clean_text(s):
    """Clean and normalize text"""
    if pd.isna(s):
        return s
    
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)  # Remove extra whitespace
    
    return s

print("🧽 Cleaning and normalizing text...")

# Clean Sanad lists
df['Sanad'] = df['Sanad'].apply(
    lambda lst: [clean_text(item) if isinstance(item, str) else item for item in lst] 
    if isinstance(lst, list) else lst
)

# Clean other text columns
text_columns = ['Hadith', 'Book', 'Matn']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

print("✅ Text cleaning completed")

print(f"📊 Sample cleaned Sanad:")
print(f"   {df['Sanad'].iloc[0]}")

## Save Cleaned Data

In [ ]:
# Create processed directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/sanadset_cleaned.csv"

print(f"💾 Saving cleaned data to: {output_path}")
print(f"📊 Final dataset shape: {df.shape}")

df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("✅ Cleaned data saved successfully!")
print(f"📁 File location: {os.path.abspath(output_path)}")

# Display final summary
print(f"\n📊 PREPROCESSING SUMMARY:")
print(f"   • Final rows: {len(df)}")
print(f"   • Columns: {len(df.columns)}")
print(f"   • Output file: {output_path}")